# LazyAd — Phase 1, Spike S1: Colab Runtime Diagnostic

**Source of Truth:** `Docs/PHASE 0 FEASIBILITY & REQUIREMENTS.md` (Approved v2.2)

### Objective
Empirically inspect the active Google Colab runtime and record the exact environment in which later LazyAd spikes will execute.

### Pass/Fail Criteria (from Phase 0 v2.2)
- **Pass:** Diagnostic executes cleanly, logs all required hardware/software specifications, and confirms active GPU availability.
- **Fail:** No GPU allocated, CUDA unavailable, or diagnostic encounters an unhandled runtime error.

> **Strict Scope Boundary:** This notebook performs **only** hardware and environment inspection. It does **not** install or load FLUX, SDXL, TripoSR, or TRELLIS, and does **not** generate images or 3D models.

## 1. Quick GPU Shell Verification (`nvidia-smi`)
Prints driver version, assigned GPU hardware name, and power/memory state.

In [ ]:
!nvidia-smi

## 2. Comprehensive Runtime & Environment Inspection
Inspects OS, Python, PyTorch, CUDA, VRAM, RAM, disk capacity, and checks for pre-installed baseline tools (`ffmpeg`, `git`, `wget`, etc.). Exports raw metrics to `/content/s1_diagnostic_report.json`.

In [ ]:
import sys
import os
import json
import shutil
import platform
import psutil
import torch

print("=" * 72)
print("LAZYAD — SPIKE S1: RUNTIME DIAGNOSTIC INSPECTION")
print("=" * 72)

# -------------------------------------------------------------------------
# A. Operating System & Core Environment
# -------------------------------------------------------------------------
os_name = platform.system()
os_release = platform.release()
os_platform = platform.platform()
python_version = platform.python_version()

# Check Linux distribution details if on Linux
linux_distro = "N/A"
if os.path.exists("/etc/os-release"):
    try:
        with open("/etc/os-release") as f:
            for line in f:
                if line.startswith("PRETTY_NAME="):
                    linux_distro = line.split("=")[1].strip().strip('"')
                    break
    except Exception:
        pass

print("\n[1. OPERATING SYSTEM & PYTHON]")
print(f"  • OS Platform         : {os_platform}")
print(f"  • Linux Distribution  : {linux_distro}")
print(f"  • Python Version      : {python_version}")

# -------------------------------------------------------------------------
# B. GPU & CUDA State Inspection
# -------------------------------------------------------------------------
gpu_available = torch.cuda.is_available()
cuda_available = torch.version.cuda is not None

gpu_count = 0
gpu_model = "None"
gpu_vram_gb = 0.0
gpu_vram_bytes = 0
compute_capability = "None"

print("\n[2. GPU & ACCELERATOR STATUS]")
if gpu_available and cuda_available:
    gpu_count = torch.cuda.device_count()
    gpu_model = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_bytes = gpu_props.total_memory
    gpu_vram_gb = gpu_vram_bytes / (1024 ** 3)
    cc = torch.cuda.get_device_capability(0)
    compute_capability = f"{cc[0]}.{cc[1]}"
    
    print("  • GPU State           : [GPU AVAILABLE]")
    print("  • CUDA State          : [CUDA AVAILABLE]")
    print(f"  • CUDA Version        : {torch.version.cuda}")
    print(f"  • cuDNN Version       : {torch.backends.cudnn.version()}")
    print(f"  • PyTorch Version     : {torch.__version__}")
    print(f"  • GPU Count           : {gpu_count}")
    print(f"  • GPU Model Name      : {gpu_model}")
    print(f"  • Total VRAM          : {gpu_vram_gb:.2f} GB ({gpu_vram_bytes:,} bytes)")
    print(f"  • Compute Capability  : {compute_capability}")
else:
    print("  • GPU State           : [GPU UNAVAILABLE]")
    print(f"  • CUDA State          : [{'CUDA AVAILABLE' if cuda_available else 'CUDA UNAVAILABLE'}]")
    print(f"  • PyTorch Version     : {torch.__version__}")
    print("  ⚠️ ACTION REQUIRED    : In Google Colab menu, navigate to: Runtime -> Change runtime type -> Select T4 GPU or A100 GPU")

# -------------------------------------------------------------------------
# C. System RAM & Ephemeral Disk Space
# -------------------------------------------------------------------------
ram = psutil.virtual_memory()
total_ram_gb = ram.total / (1024 ** 3)
avail_ram_gb = ram.available / (1024 ** 3)
used_ram_gb = (ram.total - ram.available) / (1024 ** 3)
ram_pct = ram.percent

content_dir = "/content" if os.path.exists("/content") else "."
disk = shutil.disk_usage(content_dir)
total_disk_gb = disk.total / (1024 ** 3)
free_disk_gb = disk.free / (1024 ** 3)
used_disk_gb = disk.used / (1024 ** 3)
disk_pct = (disk.used / disk.total) * 100

print("\n[3. SYSTEM MEMORY & STORAGE]")
print(f"  • Total System RAM    : {total_ram_gb:.2f} GB")
print(f"  • Available RAM       : {avail_ram_gb:.2f} GB ({100 - ram_pct:.1f}% free)")
print(f"  • Used RAM            : {used_ram_gb:.2f} GB ({ram_pct:.1f}% used)")
print(f"  • Local Disk Path     : {os.path.abspath(content_dir)}")
print(f"  • Total Disk Space    : {total_disk_gb:.2f} GB")
print(f"  • Free Disk Space     : {free_disk_gb:.2f} GB ({100 - disk_pct:.1f}% free)")

# -------------------------------------------------------------------------
# D. Baseline Tools & Pre-installed Libraries (No heavy models installed)
# -------------------------------------------------------------------------
binaries_to_check = ["ffmpeg", "ffprobe", "git", "wget", "curl", "tar", "gzip"]
binary_status = {}
for b in binaries_to_check:
    binary_path = shutil.which(b)
    binary_status[b] = binary_path if binary_path else "MISSING"

python_libs_to_check = [
    "torch", "torchvision", "torchaudio", "transformers", "diffusers", 
    "trimesh", "soundfile", "scipy", "PIL", "numpy", "google.genai"
]
lib_status = {}
for lib in python_libs_to_check:
    try:
        mod = __import__(lib)
        lib_status[lib] = getattr(mod, "__version__", "Installed")
    except ImportError:
        lib_status[lib] = "Not pre-installed (will install in later spike if needed)"

print("\n[4. BASELINE SYSTEM TOOLS]")
for b, status in binary_status.items():
    print(f"  • {b:<12} : {status}")

print("\n[5. PRE-INSTALLED PYTHON LIBRARIES]")
for lib, status in lib_status.items():
    print(f"  • {lib:<14} : {status}")

# -------------------------------------------------------------------------
# E. Export Raw Diagnostic Record
# -------------------------------------------------------------------------
diagnostic_data = {
    "spike": "S1",
    "os": {
        "platform": os_platform,
        "distro": linux_distro,
        "python": python_version
    },
    "gpu": {
        "gpu_available": gpu_available,
        "cuda_available": cuda_available,
        "cuda_version": torch.version.cuda if cuda_available else None,
        "cudnn_version": torch.backends.cudnn.version() if gpu_available else None,
        "pytorch_version": torch.__version__,
        "gpu_count": gpu_count,
        "gpu_model": gpu_model,
        "vram_gb": round(gpu_vram_gb, 2),
        "vram_bytes": gpu_vram_bytes,
        "compute_capability": compute_capability
    },
    "resources": {
        "total_ram_gb": round(total_ram_gb, 2),
        "available_ram_gb": round(avail_ram_gb, 2),
        "total_disk_gb": round(total_disk_gb, 2),
        "free_disk_gb": round(free_disk_gb, 2)
    },
    "system_binaries": binary_status,
    "python_libraries": lib_status
}

output_json_path = os.path.join(content_dir, "s1_diagnostic_report.json")
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(diagnostic_data, f, indent=2)

print(f"\n[Raw diagnostic JSON saved to: {output_json_path}]")
print("=" * 72)


## 3. Evaluated Pass / Fail Verdict (Phase 0 v2.2 Criteria)
Evaluates whether the active Colab runtime satisfies the Phase 0 v2.2 pass criteria for Spike S1.

In [ ]:
print("=" * 72)
print("SPIKE S1: EVALUATION & VERDICT")
print("=" * 72)

# Check evaluation criteria
diagnostic_completed = True
gpu_detected = torch.cuda.is_available() and (torch.cuda.device_count() > 0)
vram_sufficient = False

if gpu_detected:
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    model_name = torch.cuda.get_device_name(0)
    vram_sufficient = vram >= 6.0  # Minimum for TripoSR baseline

# Print explicit criteria checks
print(f"1. Diagnostic Completed Successfully : [YES]")
print(f"2. GPU Availability Check            : [{'PASS' if gpu_detected else 'FAIL'}] ({'GPU Available' if gpu_detected else 'GPU Unavailable'})")
print(f"3. CUDA Availability Check           : [{'PASS' if torch.version.cuda is not None else 'FAIL'}] ({'CUDA Available' if torch.version.cuda is not None else 'CUDA Unavailable'})")
print(f"4. Baseline VRAM Requirement (>=6 GB): [{'PASS' if vram_sufficient else 'FAIL'}] ({vram:.2f} GB detected)" if gpu_detected else "4. Baseline VRAM Requirement (>=6 GB): [FAIL] (No GPU)")

print("-" * 72)
if gpu_detected and vram_sufficient:
    print("🎯 FINAL VERDICT FOR SPIKE S1: [PASS]")
    print(f"   Runtime is validated on: {model_name} with {vram:.2f} GB VRAM.")
    print("   Ready to share output and proceed to Spike S2 (Hero Image Generation).")
else:
    print("❌ FINAL VERDICT FOR SPIKE S1: [FAIL]")
    if not gpu_detected:
        print("   Reason: No GPU detected in the current Colab runtime.")
        print("   Action: Go to Colab menu -> Runtime -> Change runtime type -> Hardware accelerator: T4 GPU or A100 GPU.")
    elif not vram_sufficient:
        print(f"   Reason: VRAM ({vram:.2f} GB) is below the 6.0 GB baseline threshold required for TripoSR.")
print("=" * 72)
